# Arduino OLED Bitmap Converter

Convert PNG images into SSD1306 bitmap data for Arduino OLED displays.

In [ ]:
pip install ipywidgets

In [ ]:
import numpy as np
from PIL import Image
import io
import ipywidgets as widgets
from IPython.display import display, clear_output, Javascript
import math

# =========================
# UI
# =========================

upload = widgets.FileUpload(accept='image/*', multiple=False)

screen_w_input = widgets.IntText(value=128, description="OLED W")
screen_h_input = widgets.IntText(value=64, description="OLED H")

x_input = widgets.IntText(value=0, description="X offset")
y_input = widgets.IntText(value=0, description="Y offset")

threshold = widgets.IntSlider(value=128, min=0, max=255, description="Threshold")

rotation_input = widgets.Dropdown(
    options=[("0°", 0), ("90°", 90), ("180°", 180), ("270°", 270)],
    value=0,
    description="Rotate"
)

# ✅ NEW: invert
invert_input = widgets.Checkbox(value=False, description="Invert (0/1)")

btn_display = widgets.Button(description="Display", button_style="info")
btn_gen = widgets.Button(description="Generate Bitmap", button_style="success")

btn_copy_bitmap = widgets.Button(description="Copy Bitmap", button_style="warning")
btn_copy_draw = widgets.Button(description="Copy drawBitmap", button_style="warning")

out = widgets.Output()
message = widgets.HTML("")

# =========================
# GLOBAL CACHE
# =========================

img_buffer = None
last_bitmap_text = ""
last_draw_text = ""

# =========================
# LOAD IMAGE
# =========================

def get_file():
    if not upload.value:
        return None
    v = upload.value
    return v[0] if isinstance(v, (list, tuple)) else list(v.values())[0]


def load_image():
    global img_buffer

    file = get_file()
    if file is None:
        return False

    img = Image.open(io.BytesIO(file['content'])).convert("L")

    angle = rotation_input.value
    if angle == 90:
        img = img.rotate(-90, expand=True)
    elif angle == 180:
        img = img.rotate(180, expand=True)
    elif angle == 270:
        img = img.rotate(-270, expand=True)

    img_buffer = np.array(img)
    return True


# =========================
# BUILD CANVAS (FIXED LOGIC)
# =========================

def build_canvas():
    screen_w = screen_w_input.value
    screen_h = screen_h_input.value

    sx = x_input.value
    sy = y_input.value

    # ✅ FIX: default = white background (1)
    canvas = np.ones((screen_h, screen_w), dtype=int)

    invert = invert_input.value

    for y in range(img_buffer.shape[0]):
        for x in range(img_buffer.shape[1]):

            v = 1 if img_buffer[y, x] > threshold.value else 0

            if invert:
                v = 1 - v

            nx = x + sx
            ny = y + sy

            if 0 <= nx < screen_w and 0 <= ny < screen_h:

                # ✅ FIX: draw "black pixel" as 0
                if v == 1:
                    canvas[ny, nx] = 0

    return canvas


# =========================
# COPY
# =========================

def copy_to_clipboard(text, msg):
    js = f"""
    navigator.clipboard.writeText(`{text}`)
    """
    display(Javascript(js))
    message.value = f"<span style='color:green'>✅ {msg} copied successfully</span>"


# =========================
# DISPLAY
# =========================

def on_display(_):
    with out:
        clear_output()
        message.value = ""

        if not load_image():
            print("❌ upload image first")
            return

        canvas = build_canvas()

        print(f"📺 OLED {screen_w_input.value}x{screen_h_input.value}")
        print(f"🔄 Rotate: {rotation_input.value}°")
        print(f"📍 Offset: x={x_input.value}, y={y_input.value}")
        print(f"🔁 Invert: {invert_input.value}\n")


        html = "<pre style='line-height:0.7; font-size:8px; margin:0;'>"
        for row in canvas:
            html += "".join(str(v) for v in row) + "\n"
        html += "</pre>"
        display(widgets.HTML(html))


# =========================
# GENERATE BITMAP
# =========================

def on_generate(_):
    global last_bitmap_text, last_draw_text

    with out:
        clear_output()
        message.value = ""

        if not load_image():
            print("❌ upload image first")
            return

        canvas = build_canvas()

        h, w = canvas.shape

        bitmap_lines = []
        bitmap_lines.append("const unsigned char bitmap[] PROGMEM = {")

        for row in canvas:
            row_str = "".join(str(v) for v in row)

            padded = math.ceil(len(row_str) / 8) * 8
            row_str = row_str.ljust(padded, "0")

            bytes_out = []
            for i in range(0, padded, 8):
                val = int(row_str[i:i+8], 2)
                bytes_out.append(f"0x{val:02X}")

            bitmap_lines.append("  " + ", ".join(bytes_out) + ",")

        bitmap_lines.append("};")

        last_bitmap_text = "\n".join(bitmap_lines)
        last_draw_text = f'display.drawBitmap(0, 0, bitmap, {w}, {h}, SSD1306_WHITE);'

        print(last_bitmap_text)
        print("\n👉 Arduino usage:")
        print(last_draw_text)


# =========================
# COPY EVENTS
# =========================

def on_copy_bitmap(_):
    if last_bitmap_text:
        copy_to_clipboard(last_bitmap_text, "Bitmap")


def on_copy_draw(_):
    if last_draw_text:
        copy_to_clipboard(last_draw_text, "drawBitmap")


# =========================
# EVENTS
# =========================

btn_display.on_click(on_display)
btn_gen.on_click(on_generate)
btn_copy_bitmap.on_click(on_copy_bitmap)
btn_copy_draw.on_click(on_copy_draw)


# =========================
# UI
# =========================

display(widgets.VBox([
    widgets.HTML("<h2>🔥 OLED Bitmap Editor (White BG + Black Icon + Invert)</h2>"),

    upload,
    rotation_input,

    widgets.HBox([screen_w_input, screen_h_input]),
    widgets.HBox([x_input, y_input]),

    threshold,
    invert_input,

    widgets.HBox([btn_display, btn_gen]),
    widgets.HBox([btn_copy_bitmap, btn_copy_draw]),

    message,
    out
]))